In [ ]:
%%capture
!pip install -q transformers==4.56.2 datasets==4.3.0 accelerate scikit-learn xformers

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

id2label = {0: "low", 1: "medium", 2: "hard"}
label2id = {"low": 0, "medium": 1, "hard": 2}

In [ ]:
model_name = "answerdotai/ModernBERT-large"  # verify exact repo id on HF hub before running

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    trust_remote_code=True,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
dataset = load_dataset(
    "csv",
    data_files="/content/dataset.csv"
)

dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

def preprocess(examples):
    tokens = tokenizer(
        examples["query"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokens["label"] = [label2id[label] for label in examples["label"]]
    return tokens

train_dataset = dataset["train"].map(
    preprocess, batched=True, remove_columns=dataset["train"].column_names
)
val_dataset = dataset["test"].map(
    preprocess, batched=True, remove_columns=dataset["test"].column_names
)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/944 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

In [ ]:
labels = train_dataset["labels"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([0.6946, 1.0888, 1.5578])


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    learning_rate=2e-5,
    warmup_steps=5,
    weight_decay=0.001,
    fp16=torch.cuda.is_available(),
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=0.10,
    save_strategy="no",
    lr_scheduler_type="linear",
    seed=3407,
    gradient_checkpointing=False,  # keep off — small model, small data, avoids the earlier bug class entirely
    report_to="none",
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
W0723 16:16:10.491000 1397 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
30,0.122200,0.238197,0.945148,0.941960
60,0.001800,0.322878,0.945148,0.940130
90,0.504200,0.167932,0.970464,0.967709
120,0.000100,0.205645,0.953586,0.949879
150,0.000000,0.271136,0.966245,0.962413
180,0.000600,0.160946,0.962025,0.958966
210,0.000000,0.199734,0.970464,0.967673
240,0.000000,0.161647,0.970464,0.967820
270,0.000000,0.162826,0.970464,0.967820


In [ ]:
from transformers import pipeline

sentence1 = "hi."

classifier = pipeline("text-classification", model = model,tokenizer = tokenizer)

classifier(sentence1)

Device set to use cuda:0


[{'label': 'low', 'score': 0.9999984502792358}]

In [ ]:
trainer.save_model("/content/query_classifier")
tokenizer.save_pretrained("/content/query_classifier")

('/content/query_classifier/tokenizer_config.json',
 '/content/query_classifier/special_tokens_map.json',
 '/content/query_classifier/tokenizer.json')

In [ ]:
%%capture
!pip install -U optimum[onnxruntime] onnx onnxruntime

In [ ]:
!optimum-cli export onnx \
    --model query_classifier \
    --task text-classification \
    query_classifier_onnx/

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
optimum.exporters.tasks.get_diffusers_tasks_to_model_mapping method failed to import diffusers with the error below.Please make sure you have diffusers installed and compatible with your transformers version.

Failed to import diffusers.pipelines.krea2.pipeline_krea2 because of the following error (look up to see its traceback):
cannot import name 'Qwen3VLModel' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor 

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(
    "query_classifier_onnx",
    file_name="model.onnx",
)

tokenizer = AutoTokenizer.from_pretrained("query_classifier_onnx")

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
optimum.exporters.tasks.get_diffusers_tasks_to_model_mapping method failed to import diffusers with the error below.Please make sure you have diffusers installed and compatible with your transformers version.

Failed to import diffusers.pipelines.krea2.pipeline_krea2 because of the following error (look up to see its traceback):
cannot import name 'Qwen3VLModel' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)


In [ ]:
from transformers import pipeline
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(
    "query_classifier_onnx",
    provider="CPUExecutionProvider",
)

tokenizer = AutoTokenizer.from_pretrained("query_classifier_onnx")

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=-1,   # Force CPU
)

print(classifier("hi"))
print(classifier("Can you summarize this document?"))

Device set to use cpu


[{'label': 'low', 'score': 0.9999922513961792}]
[{'label': 'low', 'score': 0.9962161183357239}]


In [ ]:
import os
os.makedirs("query_classifier_onnx_int8_v2", exist_ok=True)
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="query_classifier_onnx/model.onnx",
    model_output="query_classifier_onnx_int8_v2/model.onnx",
    weight_type=QuantType.QUInt8,
)

  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__6"
    }
  }
}
.


In [ ]:
import shutil

for f in [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]:
    shutil.copy(
        f"query_classifier_onnx/{f}",
        f"query_classifier_onnx_int8_v2/{f}",
    )

In [ ]:
from transformers import pipeline, AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification

model = ORTModelForSequenceClassification.from_pretrained(
    "query_classifier_onnx_int8_v2",
    file_name="model.onnx",
    provider="CPUExecutionProvider",
)

tokenizer = AutoTokenizer.from_pretrained("query_classifier_onnx_int8_v2")

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=-1,
)

print(classifier("hi"))
print(classifier("build the quantization project"))

Device set to use cpu


[{'label': 'low', 'score': 0.9981563687324524}]
[{'label': 'medium', 'score': 0.9481416940689087}]


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(
    "query_classifier_onnx_int8_v2",
    file_name="model.onnx",
    provider="CPUExecutionProvider",
)

tokenizer = AutoTokenizer.from_pretrained("query_classifier_onnx_int8_v2")

print("Loaded successfully!")

Loaded successfully!


In [ ]:
import shutil

shutil.make_archive(
    "query_classifier_onnx_int8_v2",
    "zip",s
    "query_classifier_onnx_int8_v2"
)

'/content/query_classifier_onnx_int8_v2.zip'

In [ ]:
from google.colab import files

files.download("query_classifier_onnx_int8_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>